# Simple Gripper Control Test
- **LT**: Open gripper
- **RT**: Close gripper
- **Press Ctrl+C** to stop

In [3]:
#!/usr/bin/env python3
# -*-coding:utf8-*-
# Go to home position
import time
from piper_sdk import C_PiperInterface_V2

if __name__ == "__main__":
    print("=== Piper Arm Go Home Script ===")

    # Initialize and connect
    piper = C_PiperInterface_V2("can0")
    piper.ConnectPort()
    time.sleep(0.5)

    # Check current status
    status = piper.GetArmStatus()
    print(f"Current control mode: {status.arm_status.ctrl_mode}")
    print(f"Arm status: {status.arm_status.arm_status}")

    # Clear emergency stop if arm is in that state
    if status.arm_status.arm_status == 1:  # EMERGENCY_STOP
        response = input("\nArm is in EMERGENCY_STOP state. Clear it? (y/n): ").strip().lower()
        if response == 'y':
            piper.EmergencyStop(0x02)  # Clear emergency stop
            time.sleep(1.0)
            print("Emergency stop cleared!")
        else:
            print("Cannot proceed with emergency stop active. Exiting.")
            exit(1)

    # Step 1: Set to CAN control mode
    print("\nStep 1: Setting CAN control mode...")
    for i in range(50):
        piper.MotionCtrl_2(0x01, 0x01, 30, 0x00)
        time.sleep(0.1)
        if piper.GetArmStatus().arm_status.ctrl_mode == 1:
            print("CAN mode activated!")
            break
    else:
        print("WARNING: Could not switch to CAN mode, continuing anyway...")

    # Step 2: Enable all motors
    print("Step 2: Enabling motors...")
    for i in range(100):
        piper.EnableArm(7)  # Enable all 7 motors (6 joints + gripper)
        time.sleep(0.1)
        enable_status = piper.GetArmEnableStatus()
        if all(enable_status):
            print(f"All motors enabled: {enable_status}")
            break
        if i % 10 == 0:
            print(f"Enabling... attempt {i+1}, status: {enable_status}")
    else:
        print(f"WARNING: Not all motors enabled: {enable_status}")
        print("The arm may need to be manually moved to a safe position first.")
        print("Try manually moving the arm closer to an upright position.")
        exit(1)

    # Step 3: Move to home position (all joints to 0)
    print("\nStep 3: Moving to home position...")
    print("WARNING: Arm will move! Make sure the area is clear.")
    input("Press Enter to continue or Ctrl+C to cancel...")

    home_joints = [20000, 0, 0, 0, 0, 0]  # All joints at 0 degrees

    for i in range(500):
        piper.MotionCtrl_2(0x01, 0x01, 30, 0x00)
        piper.JointCtrl(*home_joints)
        time.sleep(0.02)

        # Check if we're close to home
        joints = piper.GetArmJointMsgs().joint_state
        current = [
            joints.joint_1 / 1000,
            joints.joint_2 / 1000,
            joints.joint_3 / 1000,
            joints.joint_4 / 1000,
            joints.joint_5 / 1000,
            joints.joint_6 / 1000,
        ]

        if i % 50 == 0:
            print(f"Current position: {[round(j, 1) for j in current]} deg")

        # Check if close enough to home (within 5 degrees)
        if all(abs(c) < 5 for c in current):
            print("\n=== Arm is at home position! ===")
            break
    else:
        print("\nTimeout - arm may not have reached home position")
        joints = piper.GetArmJointMsgs().joint_state
        print(f"Final position: joint1={joints.joint_1/1000:.1f}, joint2={joints.joint_2/1000:.1f}, "
              f"joint3={joints.joint_3/1000:.1f}, joint4={joints.joint_4/1000:.1f}, "
              f"joint5={joints.joint_5/1000:.1f}, joint6={joints.joint_6/1000:.1f}")

    # Optional: Emergency stop
    response = input("\nActivate emergency stop? (y/n): ").strip().lower()
    if response == 'y':
        piper.EmergencyStop(0x01)  # Trigger emergency stop
        print("Emergency stop activated!")
    else:
        print("Emergency stop skipped.")

    print("\nDone!")


=== Piper Arm Go Home Script ===
Current control mode: STANDBY(0x0)
Arm status: EMERGENCY_STOP(0x1)
Emergency stop cleared!

Step 1: Setting CAN control mode...
CAN mode activated!
Step 2: Enabling motors...
All motors enabled: [True, True, True, True, True, True]

Step 3: Moving to home position...
Current position: [0.0, -2.0, 2.3, 0.0, 24.6, 0.0] deg
Current position: [19.9, 0.0, 0.0, 0.0, 0.0, 0.0] deg
Current position: [19.9, 0.0, 0.0, 0.0, 0.0, 0.0] deg
Current position: [19.9, 0.0, 0.0, 0.0, 0.0, 0.0] deg
Current position: [19.9, 0.0, 0.0, 0.0, 0.0, 0.0] deg
Current position: [19.9, 0.0, 0.0, 0.0, 0.0, 0.0] deg
Current position: [19.9, 0.0, 0.0, 0.0, 0.0, 0.0] deg
Current position: [19.9, 0.0, 0.0, 0.0, 0.0, 0.0] deg
Current position: [19.9, 0.0, 0.0, 0.0, 0.0, 0.0] deg
Current position: [19.9, 0.0, 0.0, 0.0, 0.0, 0.0] deg

Timeout - arm may not have reached home position
Final position: joint1=19.9, joint2=0.0, joint3=0.0, joint4=0.0, joint5=0.0, joint6=0.0
Emergency stop activ

In [35]:
import pygame
import time
from piper_sdk import *

# Initialize pygame and joystick
pygame.init()
pygame.joystick.init()

if pygame.joystick.get_count() == 0:
    print("No joystick found!")
else:
    joystick = pygame.joystick.Joystick(0)
    joystick.init()
    print(f"Joystick connected: {joystick.get_name()}")
    print(f"Axes: {joystick.get_numaxes()}, Buttons: {joystick.get_numbuttons()}")

Joystick connected: Logitech Dual Action
Axes: 4, Buttons: 12


In [38]:
# Connect to Piper arm (with proper reset cycle like playPos_new.py)
piper = C_PiperInterface_V2('can0')
piper.ConnectPort()
time.sleep(0.5)
 
# FULL RESET CYCLE (trigger e-stop first, then clear)
print("Resetting arm...")
piper.EmergencyStop(0x01)  # Trigger e-stop first
time.sleep(1.0)
piper.EmergencyStop(0x02)  # Then clear it
time.sleep(1.0)

# Wait for CAN mode (ctrl_mode == 1)
print("Waiting for CAN mode...")
timeout = time.time() + 10.0
while piper.GetArmStatus().arm_status.ctrl_mode != 1:
    if time.time() > timeout:
        print("ERROR: CAN mode switch failed")
        break
    piper.ModeCtrl(0x01, 0x01, 50, 0x00)
    time.sleep(0.1)

# Enable with longer timeout and slower polling
print("Enabling arm...")
enabled = False
timeout = time.time() + 10.0
while time.time() < timeout:
    if piper.EnablePiper():
        enabled = True
        print("Enabled!")
        break
    time.sleep(0.1)  # Slower polling (0.1s, not 0.02s)

if not enabled:
    print("Warning: EnablePiper timed out, continuing anyway...")

piper.ModeCtrl(0x01, 0x01, 50, 0x00)
time.sleep(0.3)

status = piper.GetArmStatus()
print(f"Arm status: {status.arm_status.arm_status}")
print(f"Ctrl mode: {status.arm_status.ctrl_mode}")
print("Ready!")

[2026-01-12 12:25:30] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] 0x150 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))


Resetting arm...


[2026-01-12 12:25:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] 0x150 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:32] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] EnableArm send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:32] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] 0x151 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))


Waiting for CAN mode...
Enabling arm...
Enabled!
Arm status: NORMAL(0x0)
Ctrl mode: CAN_CTRL(0x1)
Ready!


In [39]:
# Full movement control + gripper + speed adjust + home
print("Movement Control:")
print("  D-pad UP/DOWN        (Joint 3 - elbow)")
print("  D-pad LEFT/RIGHT     (Joint 1 - base rotation)")
print("  LB = FORWARD, RB = BACKWARD (Joint 2 - shoulder)")
print("  LT = GRIPPER OPEN, RT = GRIPPER CLOSE")
print("  Y = SPEED UP, A = SPEED DOWN")
print("  X = GO HOME (all joints to 0)")

# Re-enable arm (REQUIRED for JointCtrl to work!)
for i in range(50):
    if piper.EnablePiper():
        print("Enabled!")
        break
    time.sleep(0.02)

piper.ModeCtrl(0x01, 0x01, 30, 0x00)
time.sleep(0.2)

# Get current joints
joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1
j2 = joints.joint_state.joint_2
j3 = joints.joint_state.joint_3
j4 = joints.joint_state.joint_4
j5 = joints.joint_state.joint_5
j6 = joints.joint_state.joint_6

gripper_pos = 0
speed = 300  # Starting speed (step size)
speed_min = 100
speed_max = 1000

# Track button state for edge detection
y_was_pressed = False
a_was_pressed = False
x_was_pressed = False

print(f"Starting J1: {j1/1000:.1f}°, J2: {j2/1000:.1f}°, J3: {j3/1000:.1f}°")

try:
    while True:
        pygame.event.pump()
        
        # D-pad (hat) for UP/DOWN/LEFT/RIGHT
        hat = joystick.get_hat(0)
        dpad_left = (hat[0] == -1)
        dpad_right = (hat[0] == 1)
        dpad_up = (hat[1] == 1)
        dpad_down = (hat[1] == -1)
        
        # Bumpers and triggers
        btn_LB = joystick.get_button(4)
        btn_RB = joystick.get_button(5)
        btn_LT = joystick.get_button(6)
        btn_RT = joystick.get_button(7)
        
        # Face buttons
        btn_Y = joystick.get_button(3)
        btn_A = joystick.get_button(1)
        btn_X = joystick.get_button(0)
        
        # Speed adjust (edge detection)
        if btn_Y and not y_was_pressed:
            speed = min(speed_max, speed + 100)
        if btn_A and not a_was_pressed:
            speed = max(speed_min, speed - 100)
        y_was_pressed = btn_Y
        a_was_pressed = btn_A
        
        # Go home (edge detection)
        if btn_X and not x_was_pressed:
            j1, j2, j3, j4, j5, j6 = 0, 0, 0, 0, 0, 0
            print("\n>>> GOING HOME <<<")
        x_was_pressed = btn_X
        
        # Up/Down controls J3 (elbow)
        if dpad_up:
            j3 -= speed
        if dpad_down:
            j3 += speed
        
        # Left/Right controls J1 (base rotation)
        if dpad_left:
            j1 += speed
        if dpad_right:
            j1 -= speed
        
        # Forward/Backward controls J2 (shoulder)
        if btn_LB:
            j2 += speed
        if btn_RB:
            j2 -= speed
        
        # Gripper control
        if btn_LT:
            gripper_pos = min(70000, gripper_pos + 1500)
        if btn_RT:
            gripper_pos = max(0, gripper_pos - 1500)
        
        piper.JointCtrl(j1, j2, j3, j4, j5, j6)
        piper.GripperCtrl(gripper_pos, 1000, 0x01, 0)
        print(f"\rJ1:{j1/1000:.0f}° J2:{j2/1000:.0f}° J3:{j3/1000:.0f}° Grip:{gripper_pos/1000:.0f}mm Spd:{speed}", end="")
        time.sleep(0.02)
except KeyboardInterrupt:
    print("\nStopped")

[2026-01-12 12:25:36] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] EnableArm send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:36] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] 0x151 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))


Movement Control:
  D-pad UP/DOWN        (Joint 3 - elbow)
  D-pad LEFT/RIGHT     (Joint 1 - base rotation)
  LB = FORWARD, RB = BACKWARD (Joint 2 - shoulder)
  LT = GRIPPER OPEN, RT = GRIPPER CLOSE
  Y = SPEED UP, A = SPEED DOWN
  X = GO HOME (all joints to 0)
Enabled!
Starting J1: 4.5°, J2: 101.5°, J3: -103.7°


[2026-01-12 12:25:36] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:36] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J34 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:36] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J56 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:36] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:36] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCan

J1:4° J2:101° J3:-104° Grip:0mm Spd:300

[2026-01-12 12:25:37] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:37] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:37] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J34 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:37] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J56 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:25:37] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMe

...

J1:4° J2:101° J3:-104° Grip:0mm Spd:300

[2026-01-12 12:26:02] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J34 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:26:02] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J56 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:26:02] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:26:02] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:26:02] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J34 send failed: SendCan

J1:4° J2:101° J3:-104° Grip:0mm Spd:300

[2026-01-12 12:26:02] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:26:02] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J34 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:26:02] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J56 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:26:02] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:26:02] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCan

J1:4° J2:101° J3:-104° Grip:0mm Spd:300
Stopped


In [34]:
# Full movement control + gripper + speed adjust
print("Movement Control:")
print("  D-pad UP/DOWN        (Joint 3 - elbow)")
print("  D-pad LEFT/RIGHT     (Joint 1 - base rotation)")
print("  LB = FORWARD, RB = BACKWARD (Joint 2 - shoulder)")
print("  LT = GRIPPER OPEN, RT = GRIPPER CLOSE")
print("  Y = SPEED UP, A = SPEED DOWN")

# Re-enable arm (REQUIRED for JointCtrl to work!)
for i in range(50):
    if piper.EnablePiper():
        print("Enabled!")
        break
    time.sleep(0.02)

piper.ModeCtrl(0x01, 0x01, 30, 0x00)
time.sleep(0.2)

# Get current joints
joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1
j2 = joints.joint_state.joint_2
j3 = joints.joint_state.joint_3
j4 = joints.joint_state.joint_4
j5 = joints.joint_state.joint_5
j6 = joints.joint_state.joint_6

gripper_pos = 0
speed = 300  # Starting speed (step size)
speed_min = 100
speed_max = 1000

# Track button state for edge detection (so speed only changes once per press)
y_was_pressed = False
a_was_pressed = False

print(f"Starting J1: {j1/1000:.1f}°, J2: {j2/1000:.1f}°, J3: {j3/1000:.1f}°")

try:
    while True:
        pygame.event.pump()
        
        # D-pad (hat) for UP/DOWN/LEFT/RIGHT
        hat = joystick.get_hat(0)
        dpad_left = (hat[0] == -1)
        dpad_right = (hat[0] == 1)
        dpad_up = (hat[1] == 1)
        dpad_down = (hat[1] == -1)
        
        # Bumpers and triggers
        btn_LB = joystick.get_button(4)
        btn_RB = joystick.get_button(5)
        btn_LT = joystick.get_button(6)
        btn_RT = joystick.get_button(7)
        
        # Speed buttons
        btn_Y = joystick.get_button(3)
        btn_A = joystick.get_button(1)
        
        # Speed adjust (edge detection - only change on new press)
        if btn_Y and not y_was_pressed:
            speed = min(speed_max, speed + 100)
        if btn_A and not a_was_pressed:
            speed = max(speed_min, speed - 100)
        y_was_pressed = btn_Y
        a_was_pressed = btn_A
        
        # Up/Down controls J3 (elbow)
        if dpad_up:
            j3 -= speed
        if dpad_down:
            j3 += speed
        
        # Left/Right controls J1 (base rotation)
        if dpad_left:
            j1 += speed
        if dpad_right:
            j1 -= speed
        
        # Forward/Backward controls J2 (shoulder)
        if btn_LB:
            j2 += speed
        if btn_RB:
            j2 -= speed
        
        # Gripper control
        if btn_LT:
            gripper_pos = min(70000, gripper_pos + 1500)
        if btn_RT:
            gripper_pos = max(0, gripper_pos - 1500)
        
        piper.JointCtrl(j1, j2, j3, j4, j5, j6)
        piper.GripperCtrl(gripper_pos, 1000, 0x01, 0)
        print(f"\rJ1:{j1/1000:.0f}° J2:{j2/1000:.0f}° J3:{j3/1000:.0f}° Grip:{gripper_pos/1000:.0f}mm Speed:{speed}", end="")
        time.sleep(0.02)
except KeyboardInterrupt:
    print("\nStopped")

[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] EnableArm send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] 0x151 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))


Movement Control:
  D-pad UP/DOWN        (Joint 3 - elbow)
  D-pad LEFT/RIGHT     (Joint 1 - base rotation)
  LB = FORWARD, RB = BACKWARD (Joint 2 - shoulder)
  LT = GRIPPER OPEN, RT = GRIPPER CLOSE
  Y = SPEED UP, A = SPEED DOWN
Enabled!
Starting J1: 4.5°, J2: 101.5°, J3: -103.7°


[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J34 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J56 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCan

J1:4° J2:101° J3:-104° Grip:0mm Speed:300

[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J34 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J56 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:16] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCan

...

J1:4° J2:93° J3:-100° Grip:0mm Speed:300

[2026-01-12 12:24:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J34 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J56 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCan

J1:4° J2:93° J3:-100° Grip:0mm Speed:300

[2026-01-12 12:24:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J34 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J56 send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 12:24:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] JointCtrl_J12 send failed: SendCan

J1:4° J2:93° J3:-100° Grip:0mm Speed:300
Stopped


In [17]:
# Full movement control + gripper
print("Movement Control:")
print("  D-pad UP/DOWN        (Joint 3 - elbow)")
print("  D-pad LEFT/RIGHT     (Joint 1 - base rotation)")
print("  LB = FORWARD, RB = BACKWARD (Joint 2 - shoulder)")
print("  LT = GRIPPER OPEN, RT = GRIPPER CLOSE")

# Re-enable arm (REQUIRED for JointCtrl to work!)
for i in range(50):
    if piper.EnablePiper():
        print("Enabled!")
        break
    time.sleep(0.02)

piper.ModeCtrl(0x01, 0x01, 30, 0x00)
time.sleep(0.2)

# Get current joints
joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1
j2 = joints.joint_state.joint_2
j3 = joints.joint_state.joint_3
j4 = joints.joint_state.joint_4
j5 = joints.joint_state.joint_5
j6 = joints.joint_state.joint_6

gripper_pos = 0

print(f"Starting J1: {j1/1000:.1f}°, J2: {j2/1000:.1f}°, J3: {j3/1000:.1f}°")

try:
    while True:
        pygame.event.pump()
        
        # D-pad (hat) for UP/DOWN/LEFT/RIGHT
        hat = joystick.get_hat(0)  # Returns (x, y)
        dpad_left = (hat[0] == -1)
        dpad_right = (hat[0] == 1)
        dpad_up = (hat[1] == 1)
        dpad_down = (hat[1] == -1)
        
        # Bumpers and triggers
        btn_LB = joystick.get_button(4)  # LB = FORWARD
        btn_RB = joystick.get_button(5)  # RB = BACKWARD
        btn_LT = joystick.get_button(6)  # LT = GRIPPER OPEN
        btn_RT = joystick.get_button(7)  # RT = GRIPPER CLOSE
        
        # Up/Down controls J3 (elbow)
        if dpad_up:
            j3 -= 300
        if dpad_down:
            j3 += 300
        
        # Left/Right controls J1 (base rotation)
        if dpad_left:
            j1 += 300
        if dpad_right:
            j1 -= 300
        
        # Forward/Backward controls J2 (shoulder)
        if btn_LB:
            j2 += 300
        if btn_RB:
            j2 -= 300
        
        # Gripper control
        if btn_LT:
            gripper_pos = min(70000, gripper_pos + 1500)
        if btn_RT:
            gripper_pos = max(0, gripper_pos - 1500)
        
        piper.JointCtrl(j1, j2, j3, j4, j5, j6)
        piper.GripperCtrl(gripper_pos, 1000, 0x01, 0)
        print(f"\rJ1:{j1/1000:.0f}° J2:{j2/1000:.0f}° J3:{j3/1000:.0f}° Grip:{gripper_pos/1000:.0f}mm", end="")
        time.sleep(0.02)
except KeyboardInterrupt:
    print("\nStopped")

Movement Control:
  D-pad UP/DOWN        (Joint 3 - elbow)
  D-pad LEFT/RIGHT     (Joint 1 - base rotation)
  LB = FORWARD, RB = BACKWARD (Joint 2 - shoulder)
  LT = GRIPPER OPEN, RT = GRIPPER CLOSE
Enabled!
Starting J1: -118.6°, J2: 15.5°, J3: -59.0°
J1:-115° J2:43° J3:-110° Grip:0mmm
Stopped


In [ ]:
# Full movement control + gripper
print("Movement Control:")
print("  Y = UP,  A = DOWN     (Joint 3 - elbow)")
print("  X = LEFT, B = RIGHT   (Joint 1 - base rotation)")
print("  LB = FORWARD, RB = BACKWARD (Joint 2 - shoulder)")
print("  LT = GRIPPER OPEN, RT = GRIPPER CLOSE")

# Re-enable arm (REQUIRED for JointCtrl to work!)
print("Re-enabling arm...")
enabled = False
for i in range(50):
    # Send enable command
    piper.EnableArm(7)
    time.sleep(0.05)  # Wait for enable to take effect
    
    # Check if all motors are enabled
    enable_list = piper.GetArmEnableStatus()
    if all(enable_list):
        enabled = True
        print("✓ Enabled!")
        break
if not enabled:
    print("⚠ Warning: Arm may not be fully enabled")

piper.ModeCtrl(0x01, 0x01, 30, 0x00)
time.sleep(0.2)

# Get current joints
joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1
j2 = joints.joint_state.joint_2
j3 = joints.joint_state.joint_3
j4 = joints.joint_state.joint_4
j5 = joints.joint_state.joint_5
j6 = joints.joint_state.joint_6

gripper_pos = 0

print(f"Starting J1: {j1/1000:.1f}°, J2: {j2/1000:.1f}°, J3: {j3/1000:.1f}°")

try:
    while True:
        pygame.event.pump()
        btn_Y = joystick.get_button(3)   # Y = UP
        btn_A = joystick.get_button(1)   # A = DOWN
        btn_X = joystick.get_button(0)   # X = LEFT
        btn_B = joystick.get_button(2)   # B = RIGHT
        btn_LB = joystick.get_button(4)  # LB = FORWARD
        btn_RB = joystick.get_button(5)  # RB = BACKWARD
        btn_LT = joystick.get_button(6)  # LT = GRIPPER OPEN
        btn_RT = joystick.get_button(7)  # RT = GRIPPER CLOSE
        
        # Up/Down controls J3 (elbow)
        if btn_Y:
            j3 -= 300
        if btn_A:
            j3 += 300
        
        # Left/Right controls J1 (base rotation)
        if btn_X:
            j1 += 300
        if btn_B:
            j1 -= 300
        
        # Forward/Backward controls J2 (shoulder)
        if btn_LB:
            j2 += 300
        if btn_RB:
            j2 -= 300
        
        # Gripper control
        if btn_LT:
            gripper_pos = min(70000, gripper_pos + 1500)
        if btn_RT:
            gripper_pos = max(0, gripper_pos - 1500)
        
        piper.JointCtrl(j1, j2, j3, j4, j5, j6)
        piper.GripperCtrl(gripper_pos, 1000, 0x01, 0)
        print(f"\rJ1:{j1/1000:.0f}° J2:{j2/1000:.0f}° J3:{j3/1000:.0f}° Grip:{gripper_pos/1000:.0f}mm", end="")
        time.sleep(0.02)
except KeyboardInterrupt:
    print("\nStopped")

Movement Control:
  Y = UP,  A = DOWN     (Joint 3 - elbow)
  X = LEFT, B = RIGHT   (Joint 1 - base rotation)
  LB = FORWARD, RB = BACKWARD (Joint 2 - shoulder)
  LT = GRIPPER OPEN, RT = GRIPPER CLOSE
Enabled!
Starting J1: -116.5°, J2: 0.0°, J3: -28.3°
J1:-119° J2:16° J3:-59° Grip:0mmmm
Stopped


In [6]:
# Gripper control loop

print("Gripper Control Active!")
print("LT = Open, RT = Close")
print("Press Ctrl+C to stop\n")

gripper_pos = 0  # 0 = closed, 70000 = fully open (70mm)

try:
    while True:
        pygame.event.pump()
        
        # Logitech Dual Action: L2=button 6, R2=button 7
        lt_pressed = joystick.get_button(6) if joystick.get_numbuttons() > 6 else 0
        rt_pressed = joystick.get_button(7) if joystick.get_numbuttons() > 7 else 0
        
        # LT opens (increase), RT closes (decrease)
        if lt_pressed:
            gripper_pos = min(70000, gripper_pos + 1500)
        if rt_pressed:
            gripper_pos = max(0, gripper_pos - 1500)
        
        # Send gripper command
        # GripperCtrl(position, speed, force, mode)
        piper.GripperCtrl(gripper_pos, 1000, 0x01, 0)
        
        # Display status
        print(f"\rGripper: {gripper_pos/1000:.1f}mm | LT:{lt_pressed} RT:{rt_pressed}  ", end="")
        
        time.sleep(0.02)
        
except KeyboardInterrupt:
    print("\n\nStopped!")

[2026-01-12 11:23:30] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:30] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:30] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:30] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:30] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(

Gripper Control Active!
LT = Open, RT = Close
Press Ctrl+C to stop

Gripper: 0.0mm | LT:0 RT:0  

[2026-01-12 11:23:30] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(

Gripper: 0.0mm | LT:0 RT:0  

[2026-01-12 11:23:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:23:31] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(

...

Gripper: 0.0mm | LT:0 RT:0  

[2026-01-12 11:24:21] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:24:21] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:24:21] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:24:21] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:24:21] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(

Gripper: 0.0mm | LT:0 RT:0  

[2026-01-12 11:24:21] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:24:21] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))
[2026-01-12 11:24:21] [ERROR] [PIPER] [{'can0': <piper_sdk.interface.piper_interface_v2.C_PiperInterface_V2 object at 0xe701e8383680>}] GripperCtrl send failed: SendCanMessage(SEND_MESSAGE_FAILED (100017))


Gripper: 0.0mm | LT:0 RT:0  

Stopped!


In [15]:
# Full movement control with face buttons and bumpers
# Y = UP, A = DOWN, X = LEFT, B = RIGHT, LB = FORWARD, RB = BACKWARD
print("Movement Control:")
print("  Y = UP,  A = DOWN     (Joint 3 - elbow)")
print("  X = LEFT, B = RIGHT   (Joint 1 - base rotation)")
print("  LB = FORWARD, RB = BACKWARD (Joint 2 - shoulder)")

# Re-enable arm (REQUIRED for JointCtrl to work!)
for i in range(50):
    if piper.EnablePiper():
        print("Enabled!")
        break
    time.sleep(0.02)

piper.ModeCtrl(0x01, 0x01, 30, 0x00)
time.sleep(0.2)

# Get current joints
joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1
j2 = joints.joint_state.joint_2
j3 = joints.joint_state.joint_3
j4 = joints.joint_state.joint_4
j5 = joints.joint_state.joint_5
j6 = joints.joint_state.joint_6

print(f"Starting J1: {j1/1000:.1f}°, J2: {j2/1000:.1f}°, J3: {j3/1000:.1f}°")

try:
    while True:
        pygame.event.pump()
        btn_Y = joystick.get_button(3)   # Y = UP
        btn_A = joystick.get_button(1)   # A = DOWN
        btn_X = joystick.get_button(0)   # X = LEFT
        btn_B = joystick.get_button(2)   # B = RIGHT
        btn_LB = joystick.get_button(4)  # LB = FORWARD
        btn_RB = joystick.get_button(5)  # RB = BACKWARD
        
        # Up/Down controls J3 (elbow)
        if btn_Y:
            j3 -= 300
        if btn_A:
            j3 += 300
        
        # Left/Right controls J1 (base rotation)
        if btn_X:
            j1 += 300
        if btn_B:
            j1 -= 300
        
        # Forward/Backward controls J2 (shoulder)
        if btn_LB:
            j2 += 300  # Forward
        if btn_RB:
            j2 -= 300  # Backward
        
        piper.JointCtrl(j1, j2, j3, j4, j5, j6)
        print(f"\rJ1:{j1/1000:.0f}° J2:{j2/1000:.0f}° J3:{j3/1000:.0f}°", end="")
        time.sleep(0.02)
except KeyboardInterrupt:
    print("\nStopped")

Movement Control:
  Y = UP,  A = DOWN     (Joint 3 - elbow)
  X = LEFT, B = RIGHT   (Joint 1 - base rotation)
  LB = FORWARD, RB = BACKWARD (Joint 2 - shoulder)
Enabled!
Starting J1: -150.1°, J2: 3.7°, J3: -37.6°
J1:-116° J2:-9° J3:-28°
Stopped


In [ ]:

# Move UP/DOWN/LEFT/RIGHT with face buttons
# Y = UP, A = DOWN, X = LEFT, B = RIGHT
print("Movement Control:")
print("  Y = UP,  A = DOWN  (Joint 3 - elbow)")
print("  X = LEFT, B = RIGHT (Joint 1 - base rotation)")

# Re-enable arm (REQUIRED for JointCtrl to work!)
for i in range(50):
    if piper.EnablePiper():
        print("Enabled!")
        break
    time.sleep(0.02)

piper.ModeCtrl(0x01, 0x01, 30, 0x00)
time.sleep(0.2)

# Get current joints
joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1
j2 = joints.joint_state.joint_2
j3 = joints.joint_state.joint_3
j4 = joints.joint_state.joint_4
j5 = joints.joint_state.joint_5
j6 = joints.joint_state.joint_6

print(f"Starting J1: {j1/1000:.1f} deg, J3: {j3/1000:.1f} deg")

try:
    while True:
        pygame.event.pump()
        btn_Y = joystick.get_button(3)  # Y = UP
        btn_A = joystick.get_button(1)  # A = DOWN
        btn_X = joystick.get_button(0)  # X = LEFT
        btn_B = joystick.get_button(2)  # B = RIGHT
        
        # Up/Down controls J3 (elbow)
        if btn_Y:
            j3 -= 300
        if btn_A:
            j3 += 300
        
        # Left/Right controls J1 (base rotation)
        if btn_X:
            j1 += 300  # Rotate left
        if btn_B:
            j1 -= 300  # Rotate right
        
        piper.JointCtrl(j1, j2, j3, j4, j5, j6)
        print(f"\rJ1: {j1/1000:.1f}°  J3: {j3/1000:.1f}°  [Y/A=Up/Down, X/B=Left/Right]", end="")
        time.sleep(0.02)
except KeyboardInterrupt:
    print("\nStopped")

Gripper Control Active!
LT = Open, RT = Close
Press Ctrl+C to stop

Gripper: 5.5mm | LT:0 RT:0   

Stopped!
Movement Control:
  Y = UP,  A = DOWN  (Joint 3 - elbow)
  X = LEFT, B = RIGHT (Joint 1 - base rotation)
Enabled!
Starting J1: -109.7 deg, J3: -81.8 deg
J1: -103.7°  J3: -68.0°  [Y/A=Up/Down, X/B=Left/Right]
Stopped


In [ ]:
#Test: Rotate claw using JOINT control (Joint 6)
# Left stick X = Rotate wrist (Joint 6)

print("Claw Rotation - Joint Control")
print("Left stick LEFT/RIGHT = Rotate Joint 6")
print("Ctrl+C to stop\n")

piper.ModeCtrl(0x01, 0x01, 50, 0x00)
time.sleep(0.5)

joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1 / 1000 * 57.3
j2 = joints.joint_state.joint_2 / 1000 * 57.3
j3 = joints.joint_state.joint_3 / 1000 * 57.3
j4 = joints.joint_state.joint_4 / 1000 * 57.3
j5 = joints.joint_state.joint_5 / 1000 * 57.3
j6 = joints.joint_state.joint_6 / 1000 * 57.3

print(f"Starting Joint 6: {j6:.1f}°")

SPEED = 0.5
DEADZONE = 0.15

piper.ModeCtrl(0x01, 0x01, 50, 0x00)
time.sleep(0.1)

try:
    while True:
        pygame.event.pump()
        
        stick_x = joystick.get_axis(0)
        
        if abs(stick_x) > DEADZONE:
            j6 += stick_x * SPEED
        
        cmd = [int(j1*1000), int(j2*1000), int(j3*1000), int(j4*1000), int(j5*1000), int(j6*1000)]
        piper.JointCtrl(*cmd)
        
        print(f"\rJoint 6: {j6:.1f}°  ", end="")
        time.sleep(0.02)
        
except KeyboardInterrupt:
    print("\n\nStopped!")


# # Test: Rotate the claw (wrist rotation)
# # Left stick X = Rotate left/right (RZ - yaw)
# # Ctrl+C to stop

# print("Claw Rotation Control")
# print("Left stick LEFT/RIGHT = Rotate claw")
# print("Ctrl+C to stop\n")

# # Get initial end pose
# end_pose = piper.GetArmEndPoseMsgs()
# x = end_pose.end_pose.X_axis
# y = end_pose.end_pose.Y_axis
# z = end_pose.end_pose.Z_axis
# roll = end_pose.end_pose.RX_axis
# pitch = end_pose.end_pose.RY_axis
# yaw = end_pose.end_pose.RZ_axis

# print(f"Starting rotation: Roll={roll/1000:.1f} Pitch={pitch/1000:.1f} Yaw={yaw/1000:.1f}")

# SPEED = 500  # millidegrees per loop
# DEADZONE = 0.15

# piper.ModeCtrl(0x01, 0x00, 50, 0x00)
# time.sleep(0.1)

# try:
#     while True:
#         pygame.event.pump()
        
#         stick_x = joystick.get_axis(0)
        
#         if abs(stick_x) > DEADZONE:
#             yaw += int(stick_x * SPEED)
        
#         piper.EndPoseCtrl(int(x), int(y), int(z), int(roll), int(pitch), int(yaw))
        
#         print(f"\rYaw: {yaw/1000:.1f}°  ", end="")
#         time.sleep(0.02)
        
# except KeyboardInterrupt:
#     print("\n\nStopped!")

# # Test: Rotate claw using JOINT control (Joint 6)
# # Left stick X = Rotate wrist (Joint 6)

# print("Claw Rotation - Joint Control")
# print("Left stick LEFT/RIGHT = Rotate Joint 6")
# print("Ctrl+C to stop\n")

# piper.ModeCtrl(0x01, 0x01, 50, 0x00)
# time.sleep(0.5)

# joints = piper.GetArmJointMsgs()
# j1 = joints.joint_state.joint_1 / 1000 * 57.3
# j2 = joints.joint_state.joint_2 / 1000 * 57.3
# j3 = joints.joint_state.joint_3 / 1000 * 57.3
# j4 = joints.joint_state.joint_4 / 1000 * 57.3
# j5 = joints.joint_state.joint_5 / 1000 * 57.3
# j6 = joints.joint_state.joint_6 / 1000 * 57.3

# print(f"Starting Joint 6: {j6:.1f}°")

# SPEED = 0.5
# DEADZONE = 0.15

# piper.ModeCtrl(0x01, 0x01, 50, 0x00)
# time.sleep(0.1)

# try:
#     while True:
#         pygame.event.pump()
        
#         stick_x = joystick.get_axis(0)
        
#         if abs(stick_x) > DEADZONE:
#             j6 += stick_x * SPEED
        
#         cmd = [int(j1*1000), int(j2*1000), int(j3*1000), int(j4*1000), int(j5*1000), int(j6*1000)]
#         piper.JointCtrl(*cmd)
        
#         print(f"\rJoint 6: {j6:.1f}°  ", end="")
#         time.sleep(0.02)
        
# except KeyboardInterrupt:
#     print("\n\nStopped!")


# # Test: Rotate the claw (wrist rotation)
# # Left stick X = Rotate left/right (RZ - yaw)
# # Ctrl+C to stop

# print("Claw Rotation Control")
# print("Left stick LEFT/RIGHT = Rotate claw")
# print("Ctrl+C to stop\n")

# # Test: Full XYZ movement control
# # Left stick Y  = Up/Down (Z-axis)
# # Left stick X  = Left/Right (Y-axis)
# # Right stick Y = Forward/Backward (X-axis)
# # Ctrl+C to stop

# print("Full XYZ Movement Control")
# print("Left stick:  UP/DOWN = Z | LEFT/RIGHT = Y")
# print("Right stick: UP/DOWN = X (forward/back)")
# print("Ctrl+C to stop\n")

# # Get initial end pose
# end_pose = piper.GetArmEndPoseMsgs()
# x = end_pose.end_pose.X_axis  # in micrometers
# y = end_pose.end_pose.Y_axis
# z = end_pose.end_pose.Z_axis
# rx = end_pose.end_pose.RX_axis  # in millidegrees
# ry = end_pose.end_pose.RY_axis
# rz = end_pose.end_pose.RZ_axis

# print(f"Starting: X={x/1000:.1f}mm Y={y/1000:.1f}mm Z={z/1000:.1f}mm")

# SPEED = 500  # micrometers per loop (~0.5mm)
# DEADZONE = 0.15

# # Set to pose control mode
# piper.ModeCtrl(0x01, 0x00, 50, 0x00)  # 0x00 = pose mode
# time.sleep(0.1)

# try:
#     while True:
#         pygame.event.pump()
        
#         # Left stick
#         lx = joystick.get_axis(0)  # Left/Right (Y-axis)
#         ly = joystick.get_axis(1)  # Up/Down (Z-axis)
        
#         # Right stick
#         rx = joystick.get_axis(2)  # (unused for now)
#         ry = joystick.get_axis(3)  # Forward/Backward (X-axis)
        
#         # Move Y axis (left/right)
#         if abs(lx) > DEADZONE:
#             y += int(lx * SPEED)
        
#         # Move Z axis (up/down)
#         if abs(ly) > DEADZONE:
#             z += int(-ly * SPEED)
        
#         # Move X axis (forward/backward)
#         if abs(ry) > DEADZONE:
#             x += int(-ry * SPEED)
        
#         # Send pose command
#         piper.EndPoseCtrl(int(x), int(y), int(z), int(rx), int(ry), int(rz))
        
#         print(f"\rX:{x/1000:.1f} Y:{y/1000:.1f} Z:{z/1000:.1f}mm  ", end="")
#         time.sleep(0.02)
        
# except KeyboardInterrupt:
#     print("\n\nStopped!")


# Test: Move arm UP and DOWN (Z-axis)
# Left stick Y = Move up/down
# Ctrl+C to stop

print("Arm UP/DOWN Control")
print("Left stick UP = Arm goes UP")
print("Left stick DOWN = Arm goes DOWN")
print("Ctrl+C to stop\n")

# Get initial end pose
end_pose = piper.GetArmEndPoseMsgs()
x = end_pose.end_pose.X_axis  # in micrometers
y = end_pose.end_pose.Y_axis
z = end_pose.end_pose.Z_axis
rx = end_pose.end_pose.RX_axis  # in millidegrees
ry = end_pose.end_pose.RY_axis
rz = end_pose.end_pose.RZ_axis

print(f"Starting position: X={x/1000:.1f}mm Y={y/1000:.1f}mm Z={z/1000:.1f}mm")

SPEED = 500  # micrometers per loop (~0.5mm)
DEADZONE = 0.15

# Set to pose control mode
piper.ModeCtrl(0x01, 0x00, 50, 0x00)  # 0x00 = pose mode
time.sleep(0.1)

try:
    while True:
        pygame.event.pump()
        
        # Left stick Y for up/down
        ly = joystick.get_axis(1)
        
        # Move Z axis (up = negative stick value = positive Z)
        if abs(ly) > DEADZONE:
            z += int(-ly * SPEED)
        
        # Send pose command
        piper.EndPoseCtrl(int(x), int(y), int(z), int(rx), int(ry), int(rz))
        
        print(f"\rZ: {z/1000:.1f}mm (stick: {ly:+.2f})  ", end="")
        time.sleep(0.02)
        
except KeyboardInterrupt:
    print("\n\nStopped!")
# # Get initial end pose
# end_pose = piper.GetArmEndPoseMsgs()
# x = end_pose.end_pose.X_axis
# y = end_pose.end_pose.Y_axis
# z = end_pose.end_pose.Z_axis
# roll = end_pose.end_pose.RX_axis
# pitch = end_pose.end_pose.RY_axis
# yaw = end_pose.end_pose.RZ_axis

# print(f"Starting rotation: Roll={roll/1000:.1f} Pitch={pitch/1000:.1f} Yaw={yaw/1000:.1f}")

# SPEED = 500  # millidegrees per loop
# DEADZONE = 0.15

# piper.ModeCtrl(0x01, 0x00, 50, 0x00)
# time.sleep(0.1)

# try:
#     while True:
#         pygame.event.pump()
        
#         stick_x = joystick.get_axis(0)
        
#         if abs(stick_x) > DEADZONE:
#             yaw += int(stick_x * SPEED)
        
#         piper.EndPoseCtrl(int(x), int(y), int(z), int(roll), int(pitch), int(yaw))
        
#         print(f"\rYaw: {yaw/1000:.1f}°  ", end="")
#         time.sleep(0.02)
        
# except KeyboardInterrupt:
#     print("\n\nStopped!")

In [ ]:

# # Test: Full XYZ movement control
# # Left stick Y  = Up/Down (Z-axis)
# # Left stick X  = Left/Right (Y-axis)
# # Right stick Y = Forward/Backward (X-axis)
# # Ctrl+C to stop

# print("Full XYZ Movement Control")
# print("Left stick:  UP/DOWN = Z | LEFT/RIGHT = Y")
# print("Right stick: UP/DOWN = X (forward/back)")
# print("Ctrl+C to stop\n")

# # Get initial end pose
# end_pose = piper.GetArmEndPoseMsgs()
# x = end_pose.end_pose.X_axis  # in micrometers
# y = end_pose.end_pose.Y_axis
# z = end_pose.end_pose.Z_axis
# rx = end_pose.end_pose.RX_axis  # in millidegrees
# ry = end_pose.end_pose.RY_axis
# rz = end_pose.end_pose.RZ_axis

# print(f"Starting: X={x/1000:.1f}mm Y={y/1000:.1f}mm Z={z/1000:.1f}mm")

# SPEED = 500  # micrometers per loop (~0.5mm)
# DEADZONE = 0.15

# # Set to pose control mode
# piper.ModeCtrl(0x01, 0x00, 50, 0x00)  # 0x00 = pose mode
# time.sleep(0.1)

# try:
#     while True:
#         pygame.event.pump()
        
#         # Left stick
#         lx = joystick.get_axis(0)  # Left/Right (Y-axis)
#         ly = joystick.get_axis(1)  # Up/Down (Z-axis)
        
#         # Right stick
# # Test: Full XYZ movement control
# # Left stick Y  = Up/Down (Z-axis)
# # Left stick X  = Left/Right (Y-axis)
# # Right stick Y = Forward/Backward (X-axis)
# # Ctrl+C to stop

# print("Full XYZ Movement Control")
# print("Left stick:  UP/DOWN = Z | LEFT/RIGHT = Y")
# print("Right stick: UP/DOWN = X (forward/back)")
# print("Ctrl+C to stop\n")

# # Get initial end pose
# end_pose = piper.GetArmEndPoseMsgs()
# x = end_pose.end_pose.X_axis  # in micrometers
# y = end_pose.end_pose.Y_axis
# z = end_pose.end_pose.Z_axis
# rx = end_pose.end_pose.RX_axis  # in millidegrees
# ry = end_pose.end_pose.RY_axis
# rz = end_pose.end_pose.RZ_axis

# print(f"Starting: X={x/1000:.1f}mm Y={y/1000:.1f}mm Z={z/1000:.1f}mm")

# SPEED = 500  # micrometers per loop (~0.5mm)
# DEADZONE = 0.15

# # Set to pose control mode
# piper.ModeCtrl(0x01, 0x00, 50, 0x00)  # 0x00 = pose mode
# time.sleep(0.1)

# try:
#     while True:
#         pygame.event.pump()
        
#         # Left stick
#         lx = joystick.get_axis(0)  # Left/Right (Y-axis)
#         ly = joystick.get_axis(1)  # Up/Down (Z-axis)
        
#         # Right stick
#         rx = joystick.get_axis(2)  # (unused for now)
#         ry = joystick.get_axis(3)  # Forward/Backward (X-axis)
        
#         # Move Y axis (left/right)
#         if abs(lx) > DEADZONE:
#             y += int(lx * SPEED)
        
#         # Move Z axis (up/down)
#         if abs(ly) > DEADZONE:
#             z += int(-ly * SPEED)
        
#         # Move X axis (forward/backward)
#         if abs(ry) > DEADZONE:
#             x += int(-ry * SPEED)
        
#         # Send pose command
#         piper.EndPoseCtrl(int(x), int(y), int(z), int(rx), int(ry), int(rz))
        
#         print(f"\rX:{x/1000:.1f} Y:{y/1000:.1f} Z:{z/1000:.1f}mm  ", end="")
#         time.sleep(0.02)
        
# except KeyboardInterrupt:
#     print("\n\nStopped!")


# Test: Move arm UP and DOWN (Z-axis)
# Left stick Y = Move up/down
# Ctrl+C to stop

print("Arm UP/DOWN Control")
print("Left stick UP = Arm goes UP")
print("Left stick DOWN = Arm goes DOWN")
print("Ctrl+C to stop\n")

# Get initial end pose
end_pose = piper.GetArmEndPoseMsgs()
x = end_pose.end_pose.X_axis  # in micrometers
y = end_pose.end_pose.Y_axis
z = end_pose.end_pose.Z_axis
rx = end_pose.end_pose.RX_axis  # in millidegrees
ry = end_pose.end_pose.RY_axis
rz = end_pose.end_pose.RZ_axis

print(f"Starting position: X={x/1000:.1f}mm Y={y/1000:.1f}mm Z={z/1000:.1f}mm")

SPEED = 500  # micrometers per loop (~0.5mm)
DEADZONE = 0.15

# Set to pose control mode
piper.ModeCtrl(0x01, 0x00, 50, 0x00)  # 0x00 = pose mode
time.sleep(0.1)

try:
    while True:
        pygame.event.pump()
        
        # Left stick Y for up/down
        ly = joystick.get_axis(1)
        
        # Move Z axis (up = negative stick value = positive Z)
        if abs(ly) > DEADZONE:
            z += int(-ly * SPEED)
        
        # Send pose command
        piper.EndPoseCtrl(int(x), int(y), int(z), int(rx), int(ry), int(rz))
        
        print(f"\rZ: {z/1000:.1f}mm (stick: {ly:+.2f})  ", end="")
        time.sleep(0.02)
        
except KeyboardInterrupt:
    print("\n\nStopped!")
#         rx = joystick.get_axis(2)  # (unused for now)
#         ry = joystick.get_axis(3)  # Forward/Backward (X-axis)
        
#         # Move Y axis (left/right)
#         if abs(lx) > DEADZONE:
#             y += int(lx * SPEED)
        
#         # Move Z axis (up/down)
#         if abs(ly) > DEADZONE:
#             z += int(-ly * SPEED)
        
#         # Move X axis (forward/backward)
#         if abs(ry) > DEADZONE:
#             x += int(-ry * SPEED)
        
#         # Send pose command
#         piper.EndPoseCtrl(int(x), int(y), int(z), int(rx), int(ry), int(rz))
        
#         print(f"\rX:{x/1000:.1f} Y:{y/1000:.1f} Z:{z/1000:.1f}mm  ", end="")
#         time.sleep(0.02)
        
# except KeyboardInterrupt:
#     print("\n\nStopped!")


# Test: Move arm UP and DOWN (Z-axis)
# Left stick Y = Move up/down
# Ctrl+C to stop

print("Arm UP/DOWN Control")
print("Left stick UP = Arm goes UP")
print("Left stick DOWN = Arm goes DOWN")
print("Ctrl+C to stop\n")

# Get initial end pose
end_pose = piper.GetArmEndPoseMsgs()
x = end_pose.end_pose.X_axis  # in micrometers
y = end_pose.end_pose.Y_axis
z = end_pose.end_pose.Z_axis
rx = end_pose.end_pose.RX_axis  # in millidegrees
ry = end_pose.end_pose.RY_axis
rz = end_pose.end_pose.RZ_axis

print(f"Starting position: X={x/1000:.1f}mm Y={y/1000:.1f}mm Z={z/1000:.1f}mm")

SPEED = 500  # micrometers per loop (~0.5mm)
DEADZONE = 0.15

# Set to pose control mode
piper.ModeCtrl(0x01, 0x00, 50, 0x00)  # 0x00 = pose mode
time.sleep(0.1)

try:
    while True:
        pygame.event.pump()
        
        # Left stick Y for up/down
        ly = joystick.get_axis(1)
        
        # Move Z axis (up = negative stick value = positive Z)
        if abs(ly) > DEADZONE:
            z += int(-ly * SPEED)
        
        # Send pose command
        piper.EndPoseCtrl(int(x), int(y), int(z), int(rx), int(ry), int(rz))
        
        print(f"\rZ: {z/1000:.1f}mm (stick: {ly:+.2f})  ", end="")
        time.sleep(0.02)
        
except KeyboardInterrupt:
    print("\n\nStopped!")

Arm UP/DOWN Control
Left stick UP = Arm goes UP
Left stick DOWN = Arm goes DOWN
Ctrl+C to stop

Starting position: X=45.9mm Y=19.1mm Z=161.5mm
Z: 161.5mm (stick: -0.00)  

Stopped!
Arm UP/DOWN Control
Left stick UP = Arm goes UP
Left stick DOWN = Arm goes DOWN
Ctrl+C to stop

Starting position: X=45.9mm Y=19.1mm Z=161.5mm
Z: 161.5mm (stick: -0.00)  

In [ ]:
# Cleanup
pygame.quit()
print("Done!")

In [ ]:
# Check arm status first
status = piper.GetArmStatus()
print(f"Arm status: {status.arm_status.arm_status}")
print(f"Control mode: {status.arm_status.ctrl_mode}")

In [ ]:
print("Testing connection...")
joints = piper.GetArmJointMsgs()
print(f"Joint 6 current: {joints.joint_state.joint_6 / 1000 * 57.3:.1f}°")

In [1]:
# Check arm status and get current position
status = piper.GetArmStatus()
joints = piper.GetArmJointMsgs()

print(f"Arm status: {status.arm_status.arm_status}")
print(f"Control mode: {status.arm_status.ctrl_mode}")
print(f"\nCurrent Joint 6: {joints.joint_state.joint_6 / 1000 * 57.3:.1f}°")

NameError: name 'piper' is not defined

In [ ]:
import time
from piper_sdk import *

piper = C_PiperInterface_V2('can0')
piper.ConnectPort()
time.sleep(0.5)

print("Enabling arm...")
while not piper.EnablePiper():
    time.sleep(0.01)
print("Arm enabled!")

piper.ModeCtrl(0x01, 0x01, 50, 0x00)
time.sleep(0.2)

joints = piper.GetArmJointMsgs()
j6 = joints.joint_state.joint_6 / 1000 * 57.3
print(f"Current Joint 6: {j6:.1f}°")

Enabling arm...


In [7]:
# Get all current joint positions
joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1 / 1000 * 57.3
j2 = joints.joint_state.joint_2 / 1000 * 57.3
j3 = joints.joint_state.joint_3 / 1000 * 57.3
j4 = joints.joint_state.joint_4 / 1000 * 57.3
j5 = joints.joint_state.joint_5 / 1000 * 57.3
j6 = joints.joint_state.joint_6 / 1000 * 57.3

print(f"Current joints: J1={j1:.1f} J2={j2:.1f} J3={j3:.1f} J4={j4:.1f} J5={j5:.1f} J6={j6:.1f}")

# Try to move J6 by 30 degrees
target_j6 = j6 + 30
print(f"Moving J6 to {target_j6:.1f}°...")

piper.JointCtrl(int(j1*1000), int(j2*1000), int(j3*1000), int(j4*1000), int(j5*1000), int(target_j6*1000))
time.sleep(2)

# Check new position
joints = piper.GetArmJointMsgs()
new_j6 = joints.joint_state.joint_6 / 1000 * 57.3
print(f"New J6: {new_j6:.1f}°")

Current joints: J1=2590.8 J2=-147.0 J3=102.3 J4=-82.3 J5=1507.5 J6=0.0
Moving J6 to 30.0°...
New J6: 0.0°


In [3]:
# Small gentle movement - just rotate J6 by 5 degrees
joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1 / 1000 * 57.3
j2 = joints.joint_state.joint_2 / 1000 * 57.3
j3 = joints.joint_state.joint_3 / 1000 * 57.3
j4 = joints.joint_state.joint_4 / 1000 * 57.3
j5 = joints.joint_state.joint_5 / 1000 * 57.3
j6 = joints.joint_state.joint_6 / 1000 * 57.3

print(f"Current J6: {j6:.1f}°")

# Slow speed, small move
piper.ModeCtrl(0x01, 0x01, 20, 0x00)  # 20% speed
time.sleep(0.2)

# Move just 5 degrees
target = j6 + 5
print(f"Moving J6 to {target:.1f}° (slow)...")
piper.JointCtrl(int(j1*1000), int(j2*1000), int(j3*1000), int(j4*1000), int(j5*1000), int(target*1000))
print("Command sent")

Current J6: -7523.1°
Moving J6 to -7518.1° (slow)...
Command sent


In [4]:
# Check if it moved
joints = piper.GetArmJointMsgs()
j6 = joints.joint_state.joint_6 / 1000 * 57.3
print(f"J6 now: {j6:.1f}°")

J6 now: -7523.1°


In [5]:
# Check raw joint values
joints = piper.GetArmJointMsgs()
print("Raw joint values (as received):")
print(f"J1: {joints.joint_state.joint_1}")
print(f"J2: {joints.joint_state.joint_2}")
print(f"J3: {joints.joint_state.joint_3}")
print(f"J4: {joints.joint_state.joint_4}")
print(f"J5: {joints.joint_state.joint_5}")
print(f"J6: {joints.joint_state.joint_6}")

Raw joint values (as received):
J1: -60576
J2: -2644
J3: 2095
J4: -22935
J5: 26044
J6: -131294


In [6]:
# Try interpreting as millidegrees (divide by 1000 = degrees)
joints = piper.GetArmJointMsgs()
print("If values are millidegrees:")
print(f"J1: {joints.joint_state.joint_1 / 1000:.1f}°")
print(f"J2: {joints.joint_state.joint_2 / 1000:.1f}°")
print(f"J3: {joints.joint_state.joint_3 / 1000:.1f}°")
print(f"J4: {joints.joint_state.joint_4 / 1000:.1f}°")
print(f"J5: {joints.joint_state.joint_5 / 1000:.1f}°")
print(f"J6: {joints.joint_state.joint_6 / 1000:.1f}°")

If values are millidegrees:
J1: -60.6°
J2: -2.6°
J3: 2.1°
J4: -22.9°
J5: 26.0°
J6: -131.3°


In [7]:
# Move J6 by 10 degrees using correct units (millidegrees)
joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1
j2 = joints.joint_state.joint_2
j3 = joints.joint_state.joint_3
j4 = joints.joint_state.joint_4
j5 = joints.joint_state.joint_5
j6 = joints.joint_state.joint_6

print(f"Current J6: {j6/1000:.1f}°")

# Add 10 degrees (10000 millidegrees)
new_j6 = j6 + 10000
print(f"Target J6: {new_j6/1000:.1f}°")

piper.ModeCtrl(0x01, 0x01, 20, 0x00)  # 20% speed
time.sleep(0.2)
piper.JointCtrl(j1, j2, j3, j4, j5, new_j6)
print("Moving...")

Current J6: -131.3°
Target J6: -121.3°
Moving...


In [8]:
# Check new position
joints = piper.GetArmJointMsgs()
print(f"J6 now: {joints.joint_state.joint_6/1000:.1f}°")

J6 now: -131.3°


In [9]:
# Check arm status
status = piper.GetArmStatus()
print(f"Arm status: {status.arm_status.arm_status}")
print(f"Ctrl mode: {status.arm_status.ctrl_mode}")

Arm status: NORMAL(0x0)
Ctrl mode: CAN_CTRL(0x1)


In [10]:
# Send commands in a loop to move J6
joints = piper.GetArmJointMsgs()
j1 = joints.joint_state.joint_1
j2 = joints.joint_state.joint_2
j3 = joints.joint_state.joint_3
j4 = joints.joint_state.joint_4
j5 = joints.joint_state.joint_5
j6 = joints.joint_state.joint_6

print(f"Starting J6: {j6/1000:.1f}°")

piper.ModeCtrl(0x01, 0x01, 30, 0x00)
time.sleep(0.1)

# Gradually move J6 by sending many small commands
for i in range(50):
    j6 += 200  # 0.2 degrees per step
    piper.JointCtrl(j1, j2, j3, j4, j5, j6)
    time.sleep(0.02)

print(f"Target J6: {j6/1000:.1f}°")

# Check actual position
time.sleep(0.5)
joints = piper.GetArmJointMsgs()
print(f"Actual J6: {joints.joint_state.joint_6/1000:.1f}°")

Starting J6: -131.3°
Target J6: -121.3°
Actual J6: -131.3°


In [11]:
# Re-enable arm
print("Re-enabling...")
enable_count = 0
while not piper.EnablePiper():
    time.sleep(0.01)
    enable_count += 1
    if enable_count > 100:
        print("Enable timeout")
        break
        
print(f"Enabled after {enable_count} tries")

piper.ModeCtrl(0x01, 0x01, 50, 0x00)
time.sleep(0.3)

# Now try moving
joints = piper.GetArmJointMsgs()
j6_start = joints.joint_state.joint_6
print(f"J6 before: {j6_start/1000:.1f}°")

# Send movement command for 2 seconds
for i in range(100):
    piper.JointCtrl(
        joints.joint_state.joint_1,
        joints.joint_state.joint_2,
        joints.joint_state.joint_3,
        joints.joint_state.joint_4,
        joints.joint_state.joint_5,
        j6_start + 10000  # +10 degrees
    )
    time.sleep(0.02)

joints = piper.GetArmJointMsgs()
print(f"J6 after: {joints.joint_state.joint_6/1000:.1f}°")

Re-enabling...
Enabled after 12 tries
J6 before: -119.4°
J6 after: -112.5°


In [ ]:
# Check what's happening
status = piper.GetArmStatus()
print(f"Arm status: {status.arm_status.arm_status}")
print(f"Ctrl mode: {status.arm_status.ctrl_mode}")

# Try clearing e-stop
piper.EmergencyStop(0x02)
time.sleep(0.5)

status = piper.GetArmStatus()
print(f"After clear: {status.arm_status.arm_status}")